# Collect and clean
**Daphnne Chacon, INFO-4617**

This notebook collects three things for 31 public figures named in news coverage of the Epstein files: daily Wikipedia pageviews, the text of their Wikipedia article at three points in time, and how many emails in the Jmail archive mention them. It then cleans them into two tidy tables in `data/clean/`.

Every Wikimedia request uses an identifying User-Agent and pauses one second. The email archive is downloaded once and queried locally.

## Part 1: Collect

In [2]:
!pip -q install duckdb

In [3]:
import json
import re
import time
from datetime import date, timedelta
from pathlib import Path
from urllib.parse import quote

import requests

DATA_DIR = Path("data")   # if you run this from the repo's notebooks/ folder, use Path("../data")
RAW = DATA_DIR / "raw"
RAW.mkdir(parents=True, exist_ok=True)

# Wikimedia asks for a User-Agent that identifies you and gives contact info
HEADERS = {"User-Agent": "CU Boulder student project (Daphnne Chacon; dach2819@colorado.edu)"}
PAUSE = 1   # seconds to wait after every request

WIKI_API = "https://en.wikipedia.org/w/api.php"
PV_BASE = ("https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/"
           "en.wikipedia/all-access/user")
EMAILS_URL = "https://data.jmail.world/v1/emails.parquet"

PV_START = date(2025, 8, 1)                  # about three months before the first release
PV_END = date.today() - timedelta(days=2)    # pageviews take a day or two to appear

# Article snapshots: just before the first release, after the releases settled, and today
SNAPSHOT_TIMES = {
    "before_release": "2025-11-11T23:59:59Z",
    "after_releases": "2026-03-01T00:00:00Z",
}

session = requests.Session()
session.headers.update(HEADERS)

def polite_get(url, params=None, **kwargs):
    """GET with an identifying User-Agent and a pause after every request."""
    r = session.get(url, params=params, timeout=60, **kwargs)
    time.sleep(PAUSE)
    return r

def save(obj, name):
    (RAW / name).write_text(json.dumps(obj, indent=1, ensure_ascii=False))

In [4]:
# Public figures whose names were widely reported in news coverage of the released files.
# Being named in the files does not mean wrongdoing; this project only measures public attention.
# "titles" are their English Wikipedia article titles (old and new titles for renamed articles).
# "pattern" is a lowercase regular expression used to find them in the email text.
PEOPLE = {
    "Donald Trump":        {"titles": ["Donald Trump"],                 "pattern": r"\btrump"},
    "Bill Clinton":        {"titles": ["Bill Clinton"],                 "pattern": r"(bill|president|william j\.?|wj) clinton"},
    "Larry Summers":       {"titles": ["Larry Summers"],                "pattern": r"(larry|lawrence) (h\.? )?summers"},
    "Steve Bannon":        {"titles": ["Steve Bannon"],                 "pattern": r"bannon"},
    "Peter Thiel":         {"titles": ["Peter Thiel"],                  "pattern": r"\bthiel\b"},
    "Elon Musk":           {"titles": ["Elon Musk"],                    "pattern": r"elon musk|\belon\b"},
    "Bill Gates":          {"titles": ["Bill Gates"],                   "pattern": r"bill gates|gates, bill"},
    "Andrew Mountbatten-Windsor": {"titles": ["Andrew Mountbatten-Windsor", "Prince Andrew, Duke of York"],
                                   "pattern": r"prince andrew|duke of york|andrew mountbatten"},
    "Sarah Ferguson":      {"titles": ["Sarah Ferguson", "Sarah, Duchess of York"],
                            "pattern": r"sarah ferguson|\bfergie\b|duchess of york"},
    "Peter Mandelson":     {"titles": ["Peter Mandelson"],              "pattern": r"mandelson"},
    "Noam Chomsky":        {"titles": ["Noam Chomsky"],                 "pattern": r"chomsky"},
    "Reid Hoffman":        {"titles": ["Reid Hoffman"],                 "pattern": r"reid hoffman"},
    "Woody Allen":         {"titles": ["Woody Allen"],                  "pattern": r"woody allen|\bwoody\b"},
    "Ehud Barak":          {"titles": ["Ehud Barak"],                   "pattern": r"\bbarak\b"},
    "Leon Black":          {"titles": ["Leon Black"],                   "pattern": r"leon black"},
    "Les Wexner":          {"titles": ["Les Wexner"],                   "pattern": r"wexner"},
    "Kathryn Ruemmler":    {"titles": ["Kathryn Ruemmler"],             "pattern": r"ruemmler"},
    "Michael Wolff":       {"titles": ["Michael Wolff (journalist)"],   "pattern": r"michael wolff|\bwolff\b"},
    "Alan Dershowitz":     {"titles": ["Alan Dershowitz"],              "pattern": r"dershowitz"},
    "Ghislaine Maxwell":   {"titles": ["Ghislaine Maxwell"],            "pattern": r"ghislaine"},
    "Howard Lutnick":      {"titles": ["Howard Lutnick"],               "pattern": r"lutnick"},
    "Jes Staley":          {"titles": ["Jes Staley"],                   "pattern": r"\bstaley\b"},
    "Glenn Dubin":         {"titles": ["Glenn Dubin"],                  "pattern": r"\bdubin\b"},
    "Joi Ito":             {"titles": ["Joi Ito"],                      "pattern": r"joi ito|\bjoi\b"},
    "Lawrence Krauss":     {"titles": ["Lawrence Krauss"],              "pattern": r"krauss"},
    "Deepak Chopra":       {"titles": ["Deepak Chopra"],                "pattern": r"chopra"},
    "Steve Tisch":         {"titles": ["Steve Tisch"],                  "pattern": r"\btisch\b"},
    "Casey Wasserman":     {"titles": ["Casey Wasserman"],              "pattern": r"\bwasserman\b"},
    "Brett Ratner":        {"titles": ["Brett Ratner"],                 "pattern": r"\bratner\b"},
    "Mette-Marit":         {"titles": ["Mette-Marit, Crown Princess of Norway"], "pattern": r"mette.?marit"},
    "Thorbjørn Jagland":   {"titles": ["Thorbjørn Jagland"],            "pattern": r"jagland"},
}
save(PEOPLE, "people_list.json")
print(len(PEOPLE), "people")

31 people


In [5]:
# Step 1: resolve each title to the article's current title (follows renames and redirects)
given = sorted({t for p in PEOPLE.values() for t in p["titles"]})
resolved = {}
for i in range(0, len(given), 50):
    batch = given[i:i + 50]
    r = polite_get(WIKI_API, {"action": "query", "titles": "|".join(batch), "redirects": 1,
                              "format": "json", "formatversion": 2})
    r.raise_for_status()
    q = r.json()["query"]
    mapping = {t: t for t in batch}
    for key in ("normalized", "redirects"):
        for m in q.get(key, []):
            for t in mapping:
                if mapping[t] == m["from"]:
                    mapping[t] = m["to"]
    for page in q["pages"]:
        if page.get("missing"):
            print("WARNING, no Wikipedia article found for:", page["title"])
    resolved.update(mapping)

save(resolved, "resolved_titles.json")
for t, v in resolved.items():
    if t != v:
        print(f"{t}  ->  {v}")

Mette-Marit, Crown Princess of Norway  ->  Queen Mette-Marit of Norway
Prince Andrew, Duke of York  ->  Andrew Mountbatten-Windsor
Sarah, Duchess of York  ->  Sarah Ferguson


In [6]:
# Step 2: daily pageviews by human readers, for every title a person's article has had
pageviews = {}
for person, info in PEOPLE.items():
    titles = sorted(set(info["titles"]) | {resolved[t] for t in info["titles"]})
    pageviews[person] = {}
    for title in titles:
        article = quote(title.replace(" ", "_"), safe="")
        r = polite_get(f"{PV_BASE}/{article}/daily/{PV_START:%Y%m%d}/{PV_END:%Y%m%d}")
        if r.status_code == 404:          # no recorded views under this title
            pageviews[person][title] = []
            continue
        r.raise_for_status()
        pageviews[person][title] = r.json()["items"]
    print(person, {t: len(v) for t, v in pageviews[person].items()})

save(pageviews, "pageviews.json")

Donald Trump {'Donald Trump': 469}
Bill Clinton {'Bill Clinton': 469}
Larry Summers {'Larry Summers': 469}
Steve Bannon {'Steve Bannon': 469}
Peter Thiel {'Peter Thiel': 469}
Elon Musk {'Elon Musk': 469}
Bill Gates {'Bill Gates': 469}
Andrew Mountbatten-Windsor {'Andrew Mountbatten-Windsor': 407, 'Prince Andrew, Duke of York': 469}
Sarah Ferguson {'Sarah Ferguson': 469, 'Sarah, Duchess of York': 469}
Peter Mandelson {'Peter Mandelson': 469}
Noam Chomsky {'Noam Chomsky': 469}
Reid Hoffman {'Reid Hoffman': 469}
Woody Allen {'Woody Allen': 469}
Ehud Barak {'Ehud Barak': 469}
Leon Black {'Leon Black': 469}
Les Wexner {'Les Wexner': 469}
Kathryn Ruemmler {'Kathryn Ruemmler': 469}
Michael Wolff {'Michael Wolff (journalist)': 469}
Alan Dershowitz {'Alan Dershowitz': 469}
Ghislaine Maxwell {'Ghislaine Maxwell': 469}
Howard Lutnick {'Howard Lutnick': 469}
Jes Staley {'Jes Staley': 469}
Glenn Dubin {'Glenn Dubin': 469}
Joi Ito {'Joi Ito': 469}
Lawrence Krauss {'Lawrence Krauss': 469}
Deepak Chop

In [7]:
# Step 3: the article's text at three points in time
def get_revision(title, before=None):
    """The last revision at or before a timestamp (or the current one if before is None)."""
    params = {"action": "query", "prop": "revisions", "titles": title,
              "rvprop": "ids|timestamp|content", "rvslots": "main", "rvlimit": 1,
              "format": "json", "formatversion": 2}
    if before:
        params.update({"rvstart": before, "rvdir": "older"})
    r = polite_get(WIKI_API, params)
    r.raise_for_status()
    revs = r.json()["query"]["pages"][0].get("revisions", [])
    if not revs:
        return None
    return {"revid": revs[0]["revid"], "timestamp": revs[0]["timestamp"],
            "text": revs[0]["slots"]["main"]["content"]}

snapshots = {}
for person, info in PEOPLE.items():
    title = resolved[info["titles"][0]]
    snapshots[person] = {"title": title,
                         "before_release": get_revision(title, SNAPSHOT_TIMES["before_release"]),
                         "after_releases": get_revision(title, SNAPSHOT_TIMES["after_releases"]),
                         "now": get_revision(title)}
    print(person, "done")

save(snapshots, "article_snapshots.json")

Donald Trump done
Bill Clinton done
Larry Summers done
Steve Bannon done
Peter Thiel done
Elon Musk done
Bill Gates done
Andrew Mountbatten-Windsor done
Sarah Ferguson done
Peter Mandelson done
Noam Chomsky done
Reid Hoffman done
Woody Allen done
Ehud Barak done
Leon Black done
Les Wexner done
Kathryn Ruemmler done
Michael Wolff done
Alan Dershowitz done
Ghislaine Maxwell done
Howard Lutnick done
Jes Staley done
Glenn Dubin done
Joi Ito done
Lawrence Krauss done
Deepak Chopra done
Steve Tisch done
Casey Wasserman done
Brett Ratner done
Mette-Marit done
Thorbjørn Jagland done


In [8]:
# Step 4: the Jmail email archive, downloaded once (a single request) and kept locally
emails_path = RAW / "emails.parquet"
if not emails_path.exists():
    with session.get(EMAILS_URL, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(emails_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
print("emails.parquet:", round(emails_path.stat().st_size / 1e6, 1), "MB")

save({"collected_on": str(date.today()), "pv_start": str(PV_START), "pv_end": str(PV_END),
      "snapshot_times": SNAPSHOT_TIMES, "emails_url": EMAILS_URL}, "collection_info.json")

emails.parquet: 320.2 MB


## Part 2: Clean

In [9]:
import duckdb
import pandas as pd

CLEAN = DATA_DIR / "clean"
CLEAN.mkdir(exist_ok=True)

def load(name):
    return json.loads((RAW / name).read_text())

info = load("collection_info.json")
people_list = load("people_list.json")
resolved = load("resolved_titles.json")
pageviews = load("pageviews.json")
snapshots = load("article_snapshots.json")
print(info)

{'collected_on': '2026-11-14', 'pv_start': '2025-08-01', 'pv_end': '2026-11-12', 'snapshot_times': {'before_release': '2025-11-11T23:59:59Z', 'after_releases': '2026-03-01T00:00:00Z'}, 'emails_url': 'https://data.jmail.world/v1/emails.parquet'}


In [10]:
# Keep only the official government releases: drop the hacked Yahoo emails and promotional mail
con = duckdb.connect()
con.execute(f"""
    CREATE OR REPLACE TABLE all_emails AS
    SELECT email_drop_id,
           coalesce(is_promotional, false) AS is_promotional,
           lower(coalesce(CAST(subject AS VARCHAR), '') || ' ' ||
                 coalesce(CAST(all_participants AS VARCHAR), '') || ' ' ||
                 coalesce(CAST(content_markdown AS VARCHAR), '')) AS txt
    FROM read_parquet('{RAW / "emails.parquet"}')
""")
print(con.sql("""SELECT email_drop_id, count(*) AS emails FROM all_emails
                 GROUP BY 1 ORDER BY emails DESC""").df())

con.execute("""
    CREATE OR REPLACE TABLE emails AS
    SELECT txt FROM all_emails
    WHERE email_drop_id NOT ILIKE '%yahoo%' AND NOT is_promotional
""")
total = con.execute("SELECT count(*) FROM all_emails").fetchone()[0]
kept = con.execute("SELECT count(*) FROM emails").fetchone()[0]
print(f"All emails: {total:,} | kept after dropping Yahoo and promotional: {kept:,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

          email_drop_id  emails
0              VOL00011  667823
1              VOL00009  639582
2              VOL00010  446343
3               yahoo_2   17441
4              original    8374
5  ehud_ddos_dropsite_1    1058
6              VOL00012      71
All emails: 1,780,692 | kept after dropping Yahoo and promotional: 1,749,738


In [11]:
# Count the emails that mention each person (subject, sender/recipients, or body)
email_mentions = {}
for person, p in people_list.items():
    n = con.execute("SELECT count(*) FROM emails WHERE regexp_matches(txt, ?)",
                    [p["pattern"]]).fetchone()[0]
    email_mentions[person] = n
print(pd.Series(email_mentions).sort_values(ascending=False).to_string())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (7 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Woody Allen                   18579
Kathryn Ruemmler              15782
Joi Ito                       13703
Noam Chomsky                  12954
Lawrence Krauss               10103
Glenn Dubin                   10018
Jes Staley                     9112
Ehud Barak                     8890
Leon Black                     8153
Larry Summers                  8011
Deepak Chopra                  7411
Peter Mandelson                7392
Michael Wolff                  7188
Donald Trump                   6273
Ghislaine Maxwell              5450
Steve Bannon                   4387
Peter Thiel                    3810
Reid Hoffman                   3641
Thorbjørn Jagland              2543
Bill Gates                     2378
Elon Musk                      1682
Andrew Mountbatten-Windsor     1633
Alan Dershowitz                1056
Steve Tisch                     845
Bill Clinton                    790
Les Wexner                      615
Sarah Ferguson                  399
Howard Lutnick              

In [12]:
# Tidy daily pageviews: one row per person per day, summing old and new titles,
# and filling days the API did not report with 0 views
rows = []
for person, by_title in pageviews.items():
    for items in by_title.values():
        for item in items:
            rows.append({"person": person,
                         "date": pd.to_datetime(item["timestamp"][:8], format="%Y%m%d"),
                         "views": item["views"]})
daily = pd.DataFrame(rows, columns=["person", "date", "views"])
daily = daily.groupby(["person", "date"], as_index=False)["views"].sum()

all_days = pd.MultiIndex.from_product(
    [list(people_list), pd.date_range(info["pv_start"], info["pv_end"], freq="D")],
    names=["person", "date"])
daily = (daily.set_index(["person", "date"])
              .reindex(all_days, fill_value=0)
              .reset_index())
daily["views"] = daily["views"].astype(int)
print("Daily rows:", len(daily))
daily.head()

Daily rows: 14539


,person,date,views
0,Donald Trump,2025-08-01,50608
1,Donald Trump,2025-08-02,48437
2,Donald Trump,2025-08-03,45488
3,Donald Trump,2025-08-04,45788
4,Donald Trump,2025-08-05,46755


In [13]:
# How often each article mentions Epstein at each snapshot, and whether it has an Epstein heading
HEADING = re.compile(r"^=+[^=\n]*epstein[^=\n]*=+\s*$", re.IGNORECASE | re.MULTILINE)

def epstein_count(rev):
    return None if rev is None else rev["text"].lower().count("epstein")

rows = []
for person in people_list:
    s = snapshots[person]
    now = s["now"]
    rows.append({
        "person": person,
        "wiki_title": s["title"],
        "email_mentions": email_mentions[person],
        "epstein_mentions_before": epstein_count(s["before_release"]),
        "epstein_mentions_mar2026": epstein_count(s["after_releases"]),
        "epstein_mentions_now": epstein_count(now),
        "epstein_heading_now": bool(now and HEADING.search(now["text"])),
        "article_bytes_now": len(now["text"].encode("utf-8")) if now else None,
    })
people = pd.DataFrame(rows)
people

,person,wiki_title,email_mentions,epstein_mentions_before,epstein_mentions_mar2026,epstein_mentions_now,epstein_heading_now,article_bytes_now
0,Donald Trump,Donald Trump,6273,13,16,16,True,370435
1,Bill Clinton,Bill Clinton,790,2,22,39,True,278611
2,Larry Summers,Larry Summers,8011,26,68,62,True,84933
3,Steve Bannon,Steve Bannon,4387,20,65,65,True,341255
4,Peter Thiel,Peter Thiel,3810,32,55,40,True,379555
5,Elon Musk,Elon Musk,1682,5,53,43,True,323051
6,Bill Gates,Bill Gates,2378,41,76,107,True,242692
7,Andrew Mountbatten-Windsor,Andrew Mountbatten-Windsor,1633,154,108,125,True,243522
8,Sarah Ferguson,Sarah Ferguson,399,41,77,81,True,139367
9,Peter Mandelson,Peter Mandelson,7392,33,92,101,True,148529


In [14]:
people.to_csv(CLEAN / "people.csv", index=False)
daily.to_csv(CLEAN / "pageviews_daily.csv", index=False)
print("people.csv:", people.shape, "| pageviews_daily.csv:", daily.shape)

people.csv: (31, 8) | pageviews_daily.csv: (14539, 3)
